In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import silhouette_score

from segmentiq.clean import clean_df
from segmentiq.cluster import create_pipe
from segmentiq.data import get_data
from segmentiq.rfm import create_rfm_table
from segmentiq.score import add_scores

In [ ]:
root = Path.cwd().parents[0].resolve()
raw_path = root / Path("data/raw/Online Retail.xlsx")
cache_path = root / Path("data/cache/retail.parquet")
df = get_data(raw_path, cache_path)
cleaned_df = clean_df(df)

In [ ]:
rfm = create_rfm_table(cleaned_df)
rfm.head()

Because distributions of the RFM columns are skewed i will use log transformation then scaling in order to get normal entries for KMeans algo.

In [ ]:
inertia = []
for k in range(2, 10):
    pipe = create_pipe(n_clusters=k)
    pipe.fit(rfm)
    inertia.append(pipe["KMeans"].inertia_)
    rfm_scaled = pipe[:-1].fit_transform(rfm)
    labels = pipe.predict(rfm)
    silhouette = silhouette_score(rfm_scaled, labels)
    print(f"{k} clusters Silhouette score {silhouette}")

plt.plot(range(2, 10), inertia, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Curve")
plt.show()

We can see an elbow at 4 clusters and decent silhouette score at the same number of clusters.So i will use it 4 clusters for the champion.

In [ ]:
champion = create_pipe(n_clusters=4)
rfm_clusters = rfm.copy()
rfm_clusters["Cluster"] = champion.fit_predict(rfm)
rfm_clusters

In [ ]:
profile = rfm_clusters.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].median()
profile["Customers, %"] = (
    rfm_clusters["Cluster"].value_counts(normalize=True) * 100
).round(1)
profile["Revenue, %"] = (
    rfm_clusters.groupby("Cluster")["Monetary"].sum()
    / rfm_clusters["Monetary"].sum()
    * 100
).round(1)
profile

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 9))

for ax, column in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.barplot(x=profile.index, y=profile[column], ax=ax)
    ax.set_title(f"Median {column}")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("")

fig.suptitle("Cluster profiles")
plt.tight_layout()
plt.show()

The four clusters are readable and each of them has its own profile. I use medians here because Monetary and Frequency are still skewed inside the clusters, so the mean would be pulled up by a few big buyers.

0 - Churned - 36% of the customers and 6.6% of the revenue. They ordered once, spent around 294 pounds in total and have been inactive for half a year.

1 - Champions - 16% of the customers and 64% of the revenue. They ordered 10 times, spent 3542 pounds in total and came back 8 days ago.

2 - Average customers - 27% of the customers and 24% of the revenue. 4 orders, 1320 pounds in total, last order 51 days ago. This is the second source of money and the closest group to the Champions.

3 - New or small customers - 20% of the customers and 5% of the revenue. They bought 19 days ago, but only twice and for small money. I can not say yet if they will grow or churn, for that I would need to know when they made their first order.

The bar chart shows the same thing from another angle. Recency and Monetary are almost mirrored: the cluster with the oldest last order brings the least money. Frequency separates the Champions the most, they order 10 times while everybody else stays between 1 and 4.

The main conclusion is that 16% of the customers bring 64% of the money, so the business depends on a small group. The Average customers are the ones worth investing in, they already buy regularly and are the closest to becoming Champions.

Now I want to compare my clusters with the Bronze, Silver and Gold categories from the previous notebook. The scoring rules live in `segmentiq.score`, so I do not have to copy the bins here.

In [ ]:
scored = add_scores(rfm)
comparison = pd.crosstab(rfm_clusters["Cluster"], scored["Category"])
comparison

In [ ]:
shares = pd.crosstab(rfm_clusters["Cluster"], scored["Category"], normalize="index")
(shares * 100).round(1)

The two methods agree on the edges and argue about the middle.

Cluster 1 is 99.7% Gold, and cluster 0 is 84.5% Bronze with no Gold at all. So the best and the worst customers are the same people in both methods, and that is a good sign for my manual bins.

The interesting part is Silver. My scoring puts 903 of them into cluster 2 and 705 into cluster 3, and these two clusters look nothing alike: 4 orders and 1320 pounds against 2 orders and 409 pounds. One manual segment is really two different groups.

Gold is not one group either. 697 of them are in cluster 1, but 281 sit in cluster 2 and 46 even in cluster 3. They collected a high score from Recency and Frequency while spending average money.

I think the clusters are more honest, because I chose the bins by hand while KMeans found the borders in the data. But the categories are easier to explain to a business, and they agree where it matters most.

In the end I save the fitted pipeline. It keeps the preprocessing and the model together, so the same segments can be assigned to new customers without this notebook.

In [ ]:
model_path = root / Path("models/kmeans.joblib")
joblib.dump(champion, model_path)